In [ ]:
! pip install sewar
# wignerdpy à installer manuellement : GithHub ljbkusters/python-wigner-distribution

In [ ]:
BANDE_L9 = "TIRS2" # valeurs possibles "TIRS1", "TIRS2"
BANDE_S2 = "blue" # valeurs possibles "blue", "green", "red" et "nir"
#TYPE_ALGO = "" # valeurs possibles "DWT", "PWD" et "NSCT"

# FACTEUR DE MISE A L'ECHELLE
RESOLUTION_PLANET = 2.5
RESOLUTION_S2 = 10
RESOLUTION_L9 = 30
UPSCALE_FACTOR_SIMPLE = RESOLUTION_L9/RESOLUTION_S2 # L9.TIRS to S2
UPSCALE_FACTOR_SR_DF = RESOLUTION_L9/RESOLUTION_PLANET # L9.TIRS to S2_SR
UPSCALE_FACTOR = UPSCALE_FACTOR_SR_DF # VALEURS POSSIBLES : UPSCALE_FACTOR_SIMPLE, UPSCALE_FACTOR_SR_DF

#REPERTOIRE_DATASET = "D:\\PhD\\DATA"
REPERTOIRE_DATASET = "D:\\DATA\\FOLDER"
REPERTOIRE_SR = 'SR_' + BANDE_S2
REPERTOIRE_SATELLITE_S2_base = "S-2_DECOUPE_FINAL.OK"
REPERTOIRE_SATELLITE_S2 = REPERTOIRE_SR # VALEURS POSSIBLES : REPERTOIRE_SATELLITE_S2_base, REPERTOIRE_SR
REPERTOIRE_SATELLITE_L9 = "L-9_DECOUPE_FINAL.OK"
REPERTOIRE_RESAMPLED_L9 = "RESAMPLED_L9_SR" # VALEURS POSSIBLES : "RESAMPLED_L9_S2", "RESAMPLED_L9_SR"
REPERTOIRE_REGISTRATION_L9 = "REGISTRATION_L9_SR" # VALEURS POSSIBLES : "REGISTRATION_L9_S2", "REGISTRATION_L9_SR"
REPERTOIRE_FUSION_SIMPLE = "FUSION_SIMPLE" #(non SR)
REPERTOIRE_FUSION_SR = "FUSION_SR"
TYPE_SR = "bicubic" # VALEURS POSSIBLES : "linear", "bicubic", "cnn", "gan", "transformers", "self-algo"
REPERTOIRE_FUSION = REPERTOIRE_FUSION_SR # VALEURS POSSIBLES : REPERTOIRE_FUSION_SIMPLE, REPERTOIRE_FUSION_SR
REPERTOIRE_COMPLET_SATELLITE_S2 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE_S2
REPERTOIRE_COMPLET_SATELLITE_L9 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_SATELLITE_L9
REPERTOIRE_COMPLET_RESAMPLED_L9 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_RESAMPLED_L9
REPERTOIRE_COMPLET_REGISTRATION_L9 = REPERTOIRE_DATASET + "\\" + REPERTOIRE_REGISTRATION_L9
REPERTOIRE_COMPLET_FUSION = REPERTOIRE_DATASET + "\\" + REPERTOIRE_FUSION
EXTENSION_SENTINEL2 = ".jp2"
EXTENSION_LANDSAT9 = ".TIF"
EXTENSION_SR = '.tif'
EXTENSION_FUSION = ".tif"

print("BANDE_S2", BANDE_S2)
print("BANDE_L9", BANDE_L9)
print("UPSCALE_FACTOR", UPSCALE_FACTOR)
print("REPERTOIRE_DATASET", REPERTOIRE_DATASET)
print("REPERTOIRE_SATELLITE_S2", REPERTOIRE_SATELLITE_S2)
print("REPERTOIRE_SATELLITE_L9", REPERTOIRE_SATELLITE_L9)
print("REPERTOIRE_RESAMPLED_L9", REPERTOIRE_RESAMPLED_L9)
print("REPERTOIRE_REGISTRATION_L9", REPERTOIRE_REGISTRATION_L9)
print("REPERTOIRE_FUSION_SR", REPERTOIRE_FUSION_SR)
if REPERTOIRE_FUSION == REPERTOIRE_FUSION_SR:
    print("TYPE_SR", TYPE_SR)

In [ ]:
import rasterio
from rasterio.enums import Resampling

from skimage.transform import resize

from sewar.full_ref import rmse
from sewar.full_ref import psnr
from sewar.full_ref import scc
import pandas as pd

import os
import glob as glob
import numpy as np
import rasterio
from rasterio import plot
from rasterio.plot import show
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.cm import get_cmap
from matplotlib.axes import Axes
from IPython.display import HTML
from skimage import exposure
import rasterio
from rasterio.plot import show
import geopandas as gpd
import fiona
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sklearn
from sklearn.metrics import classification_report,accuracy_score


In [ ]:
# TEST
EXTENSION_SENTINEL2
# TIME

In [ ]:
def normalise(array):
    array_min, array_max = array.min(), array.max()
    return((array - array_min)/(array_max - array_min))

# utilisation de df.append, bien que déconseillée, car il y a peu de lignes
df_dwt_rmse_scc = pd.DataFrame(columns = ['RMSE', 'SCC'])
df_pwd_rmse_scc = pd.DataFrame(columns = ['RMSE', 'SCC'])
df_nsct_rmse_scc = pd.DataFrame(columns = ['RMSE', 'SCC'])


listes_id = os.listdir(REPERTOIRE_COMPLET_SATELLITE_L9)
for id in listes_id:
    print("Observation en cours: {}".format(id))

    ligne_dwt = []
    ligne_pwd = []
    ligne_nsct = []
    df_tmp_dwt = pd.DataFrame(columns = ['RMSE', 'SCC'])
    df_tmp_pwd = pd.DataFrame(columns = ['RMSE', 'SCC'])
    df_tmp_nsct = pd.DataFrame(columns = ['RMSE', 'SCC'])

    
    # Récupération image Landsat de base
    print("Récupération image Landsat de base")
    # liste des images du repertoire Landsat
    repertoire_image_L9 = REPERTOIRE_COMPLET_SATELLITE_L9 + "\\" + str(id)
    liste_images_sat_L9 = os.listdir(repertoire_image_L9)
    for image_L9 in liste_images_sat_L9:
        split = os.path.splitext(image_L9)
        nom_fichier = split[0]
        extension = split[1]
        if extension == EXTENSION_LANDSAT9:
            nom_fichier_decoupe = nom_fichier.split("_")
            bande_fichier = nom_fichier_decoupe[-1]
            if BANDE_L9 == "TIRS1" and bande_fichier == "B10":
                im_landsat_base = repertoire_image_L9 + "\\" + image_L9
            if BANDE_L9 == "TIRS2" and bande_fichier == "B11":
                im_landsat_base = repertoire_image_L9 + "\\" + image_L9

    # Récupération image S2 de référence
    print("Récupération image S2 de référence")
    # liste des images du repertoire Landsat
    repertoire_image_S2 = REPERTOIRE_COMPLET_SATELLITE_S2 + "\\" + str(id)
    liste_images_sat_S2 = os.listdir(repertoire_image_S2)
    for image_S2 in liste_images_sat_S2:
        split = os.path.splitext(image_S2)
        nom_fichier = split[0]
        extension = split[1]
        if extension == EXTENSION_SENTINEL2 or extension == EXTENSION_SR:
            nom_fichier_decoupe = nom_fichier.split("_")
            bande_fichier = nom_fichier_decoupe[-1]
            if REPERTOIRE_FUSION == REPERTOIRE_FUSION_SIMPLE:
                if BANDE_S2 == "blue" and bande_fichier == "B02":
                    im_reference = repertoire_image_S2 + "\\" + image_S2
                if BANDE_S2 == "green" and bande_fichier == "B03":
                    im_reference = repertoire_image_S2 + "\\" + image_S2
                if BANDE_S2 == "red" and bande_fichier == "B04":
                    im_reference = repertoire_image_S2 + "\\" + image_S2
                if BANDE_S2 == "nir" and bande_fichier == "B08":
                    im_reference = repertoire_image_S2 + "\\" + image_S2
            elif REPERTOIRE_FUSION == REPERTOIRE_FUSION_SR:
                if bande_fichier == TYPE_SR:
                    im_reference = repertoire_image_S2 + "\\" + image_S2
            else:
                    print("MAUVAIS CHOIX DE REPERTOIRE FUSION")

    rep_target_resampled = REPERTOIRE_COMPLET_RESAMPLED_L9 + "\\" + str(id)
    rep_target_coregistered = REPERTOIRE_COMPLET_REGISTRATION_L9 + "\\" + str(id)

    # création des dossier id dans les repertoires resampled et coregistered
    if not os.path.exists(rep_target_resampled):
        os.mkdir(rep_target_resampled)
    if not os.path.exists(rep_target_coregistered):
        os.mkdir(rep_target_coregistered)

    im_target_resampled = rep_target_resampled + "\\" + str(id) + "_landsat_upsampled" + EXTENSION_LANDSAT9
    im_coregistered =  rep_target_coregistered + "\\" + str(id) + "_landsat_coregistered" + EXTENSION_LANDSAT9

    
    #im_reference = "e:\\test\\s2.jp2"
    #im_landsat_base = "e:\\test\\landsat.TIF"
    #im_target_resampled = "e:\\test\\landsat_upsampled.TIF"
    #im_coregistered = "e:\\test\\landsat_coregistered.TIF"
    
    s2 = rasterio.open(im_reference)
    s2_array = s2.read()
    s2_array_norm = normalise(s2_array)
    
    
    with rasterio.open(im_landsat_base) as dataset:
    
        # resample data to target shape
        landsat_array_resampled = dataset.read(
            out_shape=(
                dataset.count,
                int(dataset.height * UPSCALE_FACTOR),
                int(dataset.width * UPSCALE_FACTOR)
            ),
            resampling=Resampling.cubic
        )
    
        # scale image transform
        transform = dataset.transform * dataset.transform.scale(
            (dataset.width / landsat_array_resampled.shape[-1]),
            (dataset.height / landsat_array_resampled.shape[-2])
        )
    
    meta = dataset.meta
    meta.update({"height": s2.meta["height"]})
    meta.update({"width": s2.meta["width"]})
    meta.update({"transform": transform})
    
    with rasterio.open(im_target_resampled, 'w', **meta) as dest:
        dest.write(landsat_array_resampled)

    
    from rasterio.warp import reproject, Resampling, calculate_default_transform
    import rasterio
    
    def reproj_match(infile, match, outfile):
        """Reproject a file to match the shape and projection of existing raster. 
        
        Parameters
        ----------
        infile : (string) path to input file to reproject
        match : (string) path to raster with desired shape and projection 
        outfile : (string) path to output file tif
        """
        # open input
        with rasterio.open(infile) as src:
            src_transform = src.transform
            
            # open input to match
            with rasterio.open(match) as match:
                dst_crs = match.crs
                
                # calculate the output transform matrix
                dst_transform, dst_width, dst_height = calculate_default_transform(
                    src.crs,     # input CRS
                    dst_crs,     # output CRS
                    match.width,   # input width
                    match.height,  # input height 
                    *match.bounds,  # unpacks input outer boundaries (left, bottom, right, top)
                )
    
            # set properties for output
            dst_kwargs = src.meta.copy()
            dst_kwargs.update({"crs": dst_crs,
                               "transform": dst_transform,
                               "width": dst_width,
                               "height": dst_height,
                               "nodata": 0})
            print("Coregistered to shape:", dst_height,dst_width,'\n Affine',dst_transform)
            # open output
            with rasterio.open(outfile, "w", **dst_kwargs) as dst:
                # iterate through bands and write using reproject function
                for i in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, i),
                        destination=rasterio.band(dst, i),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=dst_transform,
                        dst_crs=dst_crs,
                        resampling=Resampling.cubic)
    
    
    reproj_match(infile = im_reference,
                 match= im_target_resampled,
                 outfile = im_coregistered)
    
    
    landsat = rasterio.open(im_coregistered)
    landsat_array_resampled = landsat.read()
    landsat_array_resampled_norm = normalise(landsat_array_resampled)
    landsat.close()
    
    
    
    # ALGO DWT AVEC 3 METHODES DE FUSION
    import numpy as np
    import pywt
    
    # This function does the coefficient fusing according to the fusion method
    def fuseCoeff(cooef1, cooef2, method):
        if (method == 'mean'):
            cooef = (cooef1 + cooef2) / 2
        elif (method == 'min'):
            cooef = np.minimum(cooef1,cooef2)
        elif (method == 'max'):
            cooef = np.maximum(cooef1,cooef2)
        else:
            cooef = []
        return cooef
    
    # Params
    FUSION_METHOD = 'mean' # Can be 'min' || 'max || anything you choose according theory
    

    # DWT
    # Fonction pour la fusion par ondelettes
    def wavelet_fusion(image1, image2, wavelet='haar'):
        coeffs1 = pywt.dwt2(image1, wavelet)
        coeffs2 = pywt.dwt2(image2, wavelet)
    
        fused_coeffs = []
        for i in range(len(coeffs1)):
            if isinstance(coeffs1[i], tuple):
                fused_coeffs.append(tuple((fuseCoeff(coeffs1[i][j], coeffs2[i][j], FUSION_METHOD) for j in range(len(coeffs1[i])))))
            else:
                fused_coeffs.append(np.maximum(coeffs1[i], coeffs2[i]))
    
        fused_image = pywt.idwt2(fused_coeffs, wavelet)
        return np.uint8(fused_image)
    
    # Charger les images TIR et multispectrales (doivent être de la même taille)
    tir_image = landsat_array_resampled
    ms_image = s2_array
    tir_image_norm = landsat_array_resampled_norm
    ms_image_norm = s2_array_norm
    
    # Fusionner les images
    fused_image = wavelet_fusion(tir_image, ms_image)
    fused_image_norm = wavelet_fusion(tir_image_norm, ms_image_norm)
    
    # Sauvegarder l'image fusionnée
    meta = s2.meta
    meta.update({"height": fused_image.shape[1]})
    meta.update({"width": fused_image.shape[2]})

    rep_fusion_dwt = REPERTOIRE_COMPLET_FUSION + "\\" + str(id)

    if not os.path.exists(rep_fusion_dwt):
        os.mkdir(rep_fusion_dwt)

    if REPERTOIRE_FUSION == REPERTOIRE_FUSION_SIMPLE:
        img_fusion_dwt = rep_fusion_dwt + "\\" + str(id) + "_fused_dwt" + EXTENSION_FUSION
    elif REPERTOIRE_FUSION == REPERTOIRE_FUSION_SR:
        img_fusion_dwt = rep_fusion_dwt + "\\" + str(id) + "_fused_dwt_" + TYPE_SR + EXTENSION_FUSION
    else:
        print("MAUVAIS CHOIX DE REPERTOIRE DE FUSION")
    with rasterio.open(img_fusion_dwt, 'w', **meta) as dest:
        dest.write(fused_image)
    
    # Calcul des métriques
    img_source = ms_image_norm[0]
    img_fusion = resize(fused_image_norm[0], img_source.shape)
    
    rmse_fus_dwt = rmse(img_source, img_fusion)
    scc_fus_dwt = scc(img_source, img_fusion)

    # Ajout
    ligne_dwt.append(rmse_fus_dwt)
    ligne_dwt.append(scc_fus_dwt)
    df_tmp_dwt.loc[0] = ligne_dwt
    df_dwt_rmse_scc = pd.concat([df_dwt_rmse_scc, df_tmp_dwt], ignore_index = True)

    # Affichage
    print("DWT - RMSE : {} - SCC : {}".format(rmse_fus_dwt, scc_fus_dwt))
    # RMSE : 0.5056209911239881 - SCC : 0.15019552365235883
    
    
    
    # ALGO PWD
    import numpy as np
    from skimage.transform import resize
    
    import wignerdpy
    from wignerdpy.toolkits import signal_toolkit
    
    def pseudo_wigner_distribution(image):
        rows = image.shape[1]
        cols = image.shape[2]
    
        image2d = image[0]
        
        PWD = np.zeros_like(image2d, dtype=np.complex128)
    
        for row in range(rows):
            for col in range(cols):
                # Signal centered at the pixel
                signal = image2d[row, :]
                signal_shifted = np.roll(signal, col - int(cols / 2))
    
                # Compute Wigner distribution for this row
                W, max_frequency = wignerdpy.wigner_distribution(signal_shifted)
    
                PWD[row, :] += W[int(cols / 2), :]
    
        return np.abs(PWD)
    
    # Charger les images TIR et multispectrales (doivent être de la même taille)
    tir_resized = landsat_array_resampled
    ms_image = s2_array
    tir_resized_norm = landsat_array_resampled_norm
    ms_image_norm = s2_array_norm
    
    # Apply PWD to both images
    pwd_multispectral = pseudo_wigner_distribution(ms_image)
    pwd_tir = pseudo_wigner_distribution(tir_resized)
    
    fused_image = (pwd_multispectral + pwd_tir) / 2
    
    # Apply PWD to both images
    pwd_multispectral_norm = pseudo_wigner_distribution(ms_image_norm)
    pwd_tir_norm = pseudo_wigner_distribution(tir_resized_norm)
    
    fused_image_norm = (pwd_multispectral_norm + pwd_tir_norm) / 2
    
    # Sauvegarder l'image fusionnée
    meta = s2.meta
    meta.update({"height": fused_image.shape[0]})
    meta.update({"width": fused_image.shape[1]})

    rep_fusion_pwd = REPERTOIRE_COMPLET_FUSION + "\\" + str(id)

    if not os.path.exists(rep_fusion_pwd):
        os.mkdir(rep_fusion_pwd)

    if REPERTOIRE_FUSION == REPERTOIRE_FUSION_SIMPLE:
        img_fusion_pwd = rep_fusion_pwd + "\\" + str(id) + "_fused_pwd" + EXTENSION_FUSION
    elif REPERTOIRE_FUSION == REPERTOIRE_FUSION_SR:
        img_fusion_pwd = rep_fusion_pwd + "\\" + str(id) + "_fused_pwd_" + TYPE_SR + EXTENSION_FUSION
    else:
        print("MAUVAIS CHOIX DE REPERTOIRE DE FUSION")
    with rasterio.open(img_fusion_pwd, 'w', **meta) as dest:
        dest.write_band(1, fused_image.astype(rasterio.float64))
    
    # Métriques
    img_source = ms_image_norm[0]
    img_fusion = fused_image_norm
    
    rmse_fus_pwd = rmse(img_source, img_fusion)
    scc_fus_pwd = scc(img_source, img_fusion)
    
    # Ajout
    ligne_pwd.append(rmse_fus_pwd)
    ligne_pwd.append(scc_fus_pwd)
    df_tmp_pwd.loc[0] = ligne_pwd
    df_pwd_rmse_scc = pd.concat([df_pwd_rmse_scc, df_tmp_pwd], ignore_index = True)

    # Affichage
    print("PWD - RMSE : {} - SCC : {}".format(rmse_fus_pwd, scc_fus_pwd))
    # RMSE : 3.8492549114922747 - SCC : 0.05625123541493247
    
    
    
    
    # Algo NSCT 
    import pywt
    
    def wavelet_decompose(image, wavelet='coif1', level=3):
        coeffs = pywt.wavedec2(image, wavelet, level=level)
        return coeffs
    
    def fuse_coefficients(coeffs1, coeffs2):
        fused_coeffs = []
        for i in range(len(coeffs1)):
            if isinstance(coeffs1[i], tuple):
                # High-frequency bands
                cH = np.maximum(np.abs(coeffs1[i][0]), np.abs(coeffs2[i][0]))
                cV = np.maximum(np.abs(coeffs1[i][1]), np.abs(coeffs2[i][1]))
                cD = np.maximum(np.abs(coeffs1[i][2]), np.abs(coeffs2[i][2]))
                fused_coeffs.append((cH, cV, cD))
            else:
                # Low-frequency band
                fused_coeffs.append((coeffs1[i] + coeffs2[i]) / 2)
        return fused_coeffs
    
    def wavelet_reconstruct(coeffs, wavelet='coif1'):
        return pywt.waverec2(coeffs, wavelet)
    
    # Charger les images TIR et multispectrales (doivent être de la même taille)
    tir_resized = landsat_array_resampled
    ms_image = s2_array
    tir_resized_norm = landsat_array_resampled_norm
    ms_image_norm = s2_array_norm
    
    # Decompose both images
    coeffs_multispectral = wavelet_decompose(ms_image)
    coeffs_tir = wavelet_decompose(tir_resized)
    # Fuse the coefficients
    fused_coeffs = fuse_coefficients(coeffs_multispectral, coeffs_tir)
    fused_image = wavelet_reconstruct(fused_coeffs)
    
    # Decompose both images
    coeffs_multispectral_norm = wavelet_decompose(ms_image_norm)
    coeffs_tir_norm = wavelet_decompose(tir_resized_norm)
    # Fuse the coefficients
    fused_coeffs_norm = fuse_coefficients(coeffs_multispectral_norm, coeffs_tir_norm)
    fused_image_norm = wavelet_reconstruct(fused_coeffs_norm)
    
    
    # Sauvegarder l'image fusionnée
    meta = s2.meta
    meta.update({"height": fused_image.shape[1]})
    meta.update({"width": fused_image.shape[2]})
    
    rep_fusion_nsct = REPERTOIRE_COMPLET_FUSION + "\\" + str(id)

    if not os.path.exists(rep_fusion_nsct):
        os.mkdir(rep_fusion_nsct)

    if REPERTOIRE_FUSION == REPERTOIRE_FUSION_SIMPLE:
        img_fusion_nsct = rep_fusion_nsct + "\\" + str(id) + "_fused_nsct" + EXTENSION_FUSION
    elif REPERTOIRE_FUSION == REPERTOIRE_FUSION_SR:
        img_fusion_nsct = rep_fusion_nsct + "\\" + str(id) + "_fused_nsct_" + TYPE_SR + EXTENSION_FUSION
    else:
        print("MAUVAIS CHOIX DE REPERTOIRE DE FUSION")
    img_fusion_nsct = rep_fusion_nsct + "\\" + str(id) + "_fused_nsct" + EXTENSION_FUSION
    with rasterio.open(img_fusion_nsct, 'w', **meta) as dest:
        dest.write(fused_image.astype(rasterio.float64))
    
    
    img_source = ms_image_norm[0]
    img_fusion = resize(fused_image_norm[0], img_source.shape)
    
    rmse_fus_nsct = rmse(img_source, img_fusion)
    scc_fus_nsct = scc(img_source, img_fusion)
    
    # Ajout
    ligne_nsct.append(rmse_fus_nsct)
    ligne_nsct.append(scc_fus_nsct)
    df_tmp_nsct.loc[0] = ligne_nsct
    df_nsct_rmse_scc = pd.concat([df_nsct_rmse_scc, df_tmp_nsct], ignore_index = True)

    # Affichage
    print("NSCT - RMSE : {} - SCC : {}".format(rmse_fus_nsct, scc_fus_nsct))
    # RMSE : 0.2339928652913735 - SCC : 0.10472281497988677


In [ ]:
print("Résultats algo DWT -", REPERTOIRE_FUSION)
if REPERTOIRE_FUSION == REPERTOIRE_FUSION_SR:
    print(TYPE_SR)
    
df_dwt_rmse_scc.mean()

In [ ]:
print("Résultats algo PWD -", REPERTOIRE_FUSION)
if REPERTOIRE_FUSION == REPERTOIRE_FUSION_SR:
    print(TYPE_SR)
    
df_pwd_rmse_scc.mean()

In [ ]:
print("Résultats algo NSCT -", REPERTOIRE_FUSION)
if REPERTOIRE_FUSION == REPERTOIRE_FUSION_SR:
    print(TYPE_SR)

df_nsct_rmse_scc.mean()